# Notebook 00 : Configuration et utilitaires communs

## Objectif
Ce notebook définit les **constantes, chemins et fonctions utilitaires** réutilisés dans tous les notebooks suivants. Il est exporté en tant que module Python (`config.py`) pour être importé facilement.


## Contenu
- Chemins vers les données (adaptables à votre machine)
- Listes de mots-clés VSS et d'exclusion
- Fonction de regroupement en blocs idéologiques
- Couleurs et ordre des blocs pour les graphiques

## 1. Installation des dépendances

In [13]:

import subprocess, sys

packages = [
    "lxml",           # Parsing des fichiers XML de l'Assemblée
    "tqdm",           # Barres de progression
    "gensim",         # Topic Modeling (LDA) et Word2Vec
    "nltk",           # Stopwords et stemming français
    "seaborn",        # Graphiques statistiques
    "matplotlib",     # Graphiques de base
    "scikit-learn",   # Similarité cosinus
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Toutes les dépendances de base sont installées.")

Toutes les dépendances de base sont installées.


## 2. Définition des chemins

In [14]:
# ==========================================================================
# CHEMINS 
# ==========================================================================

import os

BASE_DIR = "/home/onyxia/work/projet_eco_socio"

# Dossiers des données brutes XML (comptes rendus de l'Assemblée)
DOSSIERS_LEGISLATURES = {
    "XV":   os.path.join(BASE_DIR, "data", "CompteRendusXV"),
    "XVI":  os.path.join(BASE_DIR, "data", "CompteRendusXVI"),
    "XVII": os.path.join(BASE_DIR, "data", "CompteRendusXVII"),
}

# Dossiers des séances triées (celles qui parlent de VSS)
DOSSIER_SORTED = {
    "xv":   os.path.join(BASE_DIR, "sorted", "xv"),
    "xvi":  os.path.join(BASE_DIR, "sorted", "xvi"),
    "xvii": os.path.join(BASE_DIR, "sorted", "xvii"),
}

# Dossiers des données sur les acteurs et organes
DOSSIER_PARTIS  = os.path.join(BASE_DIR, "partis")
DOSSIER_DEPUTES = os.path.join(BASE_DIR, "deputes")

# Dossier de sortie pour les DataFrames intermédiaires
DOSSIER_DATAFRAMES = os.path.join(BASE_DIR, "dataframes")
os.makedirs(DOSSIER_DATAFRAMES, exist_ok=True)

# Fichiers de sortie principaux
CHEMIN_DF_DEPUTES     = os.path.join(BASE_DIR, "dataframes", "df_deputes.csv")
CHEMIN_DF_GLOBAL      = os.path.join(BASE_DIR, "df_global.pkl")
CHEMIN_DF_VSS_PROPRE  = os.path.join(BASE_DIR, "df_vss_propre.pkl")
CHEMIN_DF_EMBEDDINGS  = os.path.join(BASE_DIR, "df_vss_embeddings.pkl")

# Dossier pour les sorties d'analyse
DOSSIER_ANALYSES = os.path.join(BASE_DIR, "analyses")
os.makedirs(DOSSIER_ANALYSES, exist_ok=True)

# API LLM (serveur Ollama de l'ENSAE)
URL_OLLAMA = "https://ollama-api.lab.groupe-genes.fr/api/chat"

print("Chemins configurés.")
print(f"   BASE_DIR = {BASE_DIR}")

Chemins configurés.
   BASE_DIR = /home/onyxia/work/projet_eco_socio


## 3. Listes de mots-clés

In [15]:
# ==========================================================================
# MOTS-CLÉS VSS (Filtre Positif)
# ==========================================================================
# Racines de mots pour capturer toutes les déclinaisons (singulier, pluriel,
# adjectif, verbe conjugué). Par exemple "viol" capte "viol", "viols",
# "violences", "violeur", etc.

A_TESTER = [
    "viol ", "sexis", "sexuel", "conjugal", "féminicide", "harcèl",
    "inceste", "outrage", "misogyn",
    "sexe", "genre", "pédocrim", "pédophil", "prostitu",
    "proxénét", "mutilation", "mariage forcé",
    "ivg", "avortement",
    "discrimination", "stéréotype", "cybersexis", "revenge porn",
    "me too", "metoo", "balancetonporc", "consentement"
]

# ==========================================================================
# MOTS D'EXCLUSION (Filtre Négatif — contre les faux positifs)
# ==========================================================================
# Certains mots-clés VSS apparaissent dans des contextes non pertinents :
# - "consentement" → consentement à l'impôt, consentement RGPD
# - "harcèlement" → harcèlement téléphonique/commercial
# - "outrage" → outrage à agent, outrage au drapeau

MOTS_EXCLUSION = [
    # Consentement fiscal et médical
    "impôt", "fiscal", "fraude", "évasion", "prélèvement", "budget",
    "don d'organe", "soins", "patient",
    # Consentement numérique (RGPD)
    "rgpd", "données personnelles", "cookie", "internet",
    # Harcèlement non-VSS
    "téléphonique", "commercial", "démarchage",
    # Outrage non-VSS
    "outrage à agent", "outrage à magistrat", "outrage au drapeau", "rébellion"
]

# Seuil de mots-clés pour retenir une séance entière lors du tri
SEUIL_TRI = 3

# ==========================================================================
# VOCABULAIRE IDENTITAIRE / MIGRATOIRE (pour l'analyse de cadrage)
# ==========================================================================

MOTS_IDENTITAIRES = [
    "immigr", "clandestin", "étranger", "migrant", "réfugié", "exilé",
    "demandeur d'asile", "sans-papier", "sans papier", "oqtf",
    "expulsion", "frontière", "reconduite", "éloignement",
    "islam", "musulman", "charia", "voile", "abaya", "burqa",
    "confession", "séparatisme", "communautarisme", "assimilation", "intégration",
    "maghrébin", "africain", "arabe", "origine étrangère", "civilisation",
    "ensauvagement", "grand remplacement", "délinquan",
]

MOTS_VSS_VIOLENCE = [
    "viol", "agress", "féminicide", "mutilation", "prostitu", "proxénét", "tournante",
    "harcel", "harcèl", "cyberharcèl", "cyber-harcèl",
    "sexuel", "sexis", "conjugal", "inceste", "pédocrimin", "pédophil",
    "patriarca", "misogyn", "machis", "emprise", "soumission", "consentement",
    "stéréotype", "domination masculine", "culture du viol", "me too", "metoo"
]

print(f"{len(A_TESTER)} mots-clés VSS, {len(MOTS_EXCLUSION)} mots d'exclusion, {len(MOTS_IDENTITAIRES)} mots identitaires.")

27 mots-clés VSS, 20 mots d'exclusion, 33 mots identitaires.


## 4. Regroupement en blocs idéologiques

In [16]:
# ==========================================================================
# BLOCS IDÉOLOGIQUES
# ==========================================================================
# On regroupe les partis politiques en 5 familles pour simplifier l'analyse.
# La fonction gère les changements de noms (ex : LREM → Renaissance).

def regrouper_blocs_ideologiques(nom_parti):
    """
    Associe un nom de parti politique à un bloc idéologique.

    Retourne None pour les partis trop petits ou inclassables
    (ex : partis ultramarins, micro-partis...), qui seront exclus de l'analyse.
    """
    nom = str(nom_parti).lower()

    # Extrême Droite
    if any(x in nom for x in ["rassemblement national", "front national", "udr"]):
        return "Extrême Droite"

    # Droite Traditionnelle
    if "républicains" in nom:
        return "Droite Traditionnelle"

    # Centre (majorité présidentielle)
    if any(c in nom for c in ["en marche", "renaissance", "ensemble", "modem",
                               "mouvement démocrate", "horizons"]):
        return "Centre"

    # Gauche Modérée
    if any(g in nom for g in ["socialiste", "écologistes", "europe écologie les verts",
                               "radical de gauche"]):
        return "Gauche Modérée"

    # Gauche Radicale
    if any(g in nom for g in ["france insoumise", "communiste", "lfi"]):
        return "Gauche Radicale"

    return None


# Couleurs et ordre pour les graphiques
ORDRE_BLOCS = [
    "Extrême Droite", "Droite Traditionnelle", "Centre",
    "Gauche Modérée", "Gauche Radicale"
]

COULEURS_BLOCS = {
    "Extrême Droite":        "#8B0000",
    "Droite Traditionnelle": "#1E3A8A",
    "Centre":                "#D97706",
    "Gauche Modérée":        "#166534",
    "Gauche Radicale":       "#DC2626",
}

print("Fonction regrouper_blocs_ideologiques() et constantes graphiques définies.")

Fonction regrouper_blocs_ideologiques() et constantes graphiques définies.


## 5. Export en module Python

In [17]:
# ==========================================================================
# EXPORT — Génère le fichier config.py importable par les autres notebooks
# ==========================================================================

config_content = '''
import os

# --- Chemins ---
BASE_DIR = "{base_dir}"

DOSSIERS_LEGISLATURES = {dossiers_leg}
DOSSIER_SORTED = {dossier_sorted}
DOSSIER_PARTIS  = "{dossier_partis}"
DOSSIER_DEPUTES = "{dossier_deputes}"
DOSSIER_DATAFRAMES = "{dossier_df}"
DOSSIER_ANALYSES = "{dossier_analyses}"

CHEMIN_DF_DEPUTES     = "{chemin_deputes}"
CHEMIN_DF_GLOBAL      = "{chemin_global}"
CHEMIN_DF_VSS_PROPRE  = "{chemin_vss}"
CHEMIN_DF_EMBEDDINGS  = "{chemin_embed}"

URL_OLLAMA = "{url_ollama}"

# --- Mots-clés ---
A_TESTER = {a_tester}
MOTS_EXCLUSION = {mots_excl}
SEUIL_TRI = {seuil}
MOTS_IDENTITAIRES = {mots_id}
MOTS_VSS_VIOLENCE = {mots_vss_v}

# --- Blocs ---
ORDRE_BLOCS = {ordre}
COULEURS_BLOCS = {couleurs}

def regrouper_blocs_ideologiques(nom_parti):
    nom = str(nom_parti).lower()
    if any(x in nom for x in ["rassemblement national", "front national", "udr"]): return "Extrême Droite"
    if "républicains" in nom: return "Droite Traditionnelle"
    if any(c in nom for c in ["en marche", "renaissance", "ensemble", "modem", "mouvement démocrate", "horizons"]): return "Centre"
    if any(g in nom for g in ["socialiste", "écologistes", "europe écologie les verts", "radical de gauche"]): return "Gauche Modérée"
    if any(g in nom for g in ["france insoumise", "communiste", "lfi"]): return "Gauche Radicale"
    return None
'''.format(
    base_dir=BASE_DIR,
    dossiers_leg=repr(DOSSIERS_LEGISLATURES),
    dossier_sorted=repr(DOSSIER_SORTED),
    dossier_partis=DOSSIER_PARTIS,
    dossier_deputes=DOSSIER_DEPUTES,
    dossier_df=DOSSIER_DATAFRAMES,
    dossier_analyses=DOSSIER_ANALYSES,
    chemin_deputes=CHEMIN_DF_DEPUTES,
    chemin_global=CHEMIN_DF_GLOBAL,
    chemin_vss=CHEMIN_DF_VSS_PROPRE,
    chemin_embed=CHEMIN_DF_EMBEDDINGS,
    url_ollama=URL_OLLAMA,
    a_tester=repr(A_TESTER),
    mots_excl=repr(MOTS_EXCLUSION),
    seuil=SEUIL_TRI,
    mots_id=repr(MOTS_IDENTITAIRES),
    mots_vss_v=repr(MOTS_VSS_VIOLENCE),
    ordre=repr(ORDRE_BLOCS),
    couleurs=repr(COULEURS_BLOCS),
)

with open("config.py", "w", encoding="utf-8") as f:
    f.write(config_content)

print(" Fichier config.py exporté avec succès.")
print("   On peut maintenant l'importer dans les autres notebooks avec :")
print('   from config import *')

 Fichier config.py exporté avec succès.
   On peut maintenant l'importer dans les autres notebooks avec :
   from config import *
